# UCInsure: Hurricane Risk Model

End-to-end sklearn pipeline using FEMA National Risk Index (NRI) census-tract data.

| # | Stage | Description |
|---|-------|-------------|
| 1 | **Setup** | Imports and constants |
| 2 | **Data Loading** | Load NRI CSV with coastal-state filter |
| 3 | **EDA** | Shape, distributions, missing values |
| 4 | **Feature Engineering** | POP_DENSITY, EXPOSURE_RATIO, normalised label |
| 5 | **Train / Test Split** | 80/20 split |
| 6 | **Model Training** | GBR + RF ensemble (sklearn, no PyTorch required) |
| 7 | **Model Evaluation** | MAE, R², residual plot |
| 8 | **Risk Scoring** | Normalised score [0-1] mapped to low/medium/high |
| 9 | **Geographic Output** | Per-tract scored CSV with lat/lon |
| 10 | **Chart** | Risk distribution by state |
| 11 | **Summary** | Stats and export paths |

## Data Source
- FEMA National Risk Index — Census Tract level
- Download: https://hazards.fema.gov/nri/data-resources
- Rename the downloaded CSV to `NRI2025.csv` and place it in `src/hurricane_model/`

## 1. Setup & Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 4)

DATA_PATH    = Path("NRI2025.csv")
MODEL_DIR    = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

HURRICANE_COLS = [
    "HRCN_EVNTS", "HRCN_AFREQ", "HRCN_EXP_AREA",
    "HRCN_EXPB",  "HRCN_EXPP",  "HRCN_HLRB", "HRCN_HLRP",
    "BUILDVALUE", "POPULATION", "AREA", "SOVI_SCORE", "RESL_SCORE",
]
LABEL_COL = "HRCN_EALB"
ID_COLS   = ["STATE", "STATEABBRV", "COUNTY", "TRACTFIPS"]
LAT_COL   = "CENTLAT"
LON_COL   = "CENTLON"

COASTAL_STATES = [
    "Florida", "Texas", "Louisiana", "Mississippi", "Alabama", "Georgia",
    "South Carolina", "North Carolina", "Virginia", "Maryland", "Delaware",
    "New Jersey", "New York", "Connecticut", "Rhode Island", "Massachusetts",
    "New Hampshire", "Maine",
]

print("Imports ready.")

Imports ready.


## 2. Data Loading

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} not found.\n"
        "Download the NRI Census Tract CSV from:\n"
        "  https://hazards.fema.gov/nri/data-resources\n"
        "Rename it to NRI2025.csv and place it next to this notebook."
    )

want_cols = ID_COLS + HURRICANE_COLS + [LABEL_COL, LAT_COL, LON_COL]
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
safe_cols = [c for c in want_cols if c in df_raw.columns]
df_raw = df_raw[safe_cols].copy()

if "STATE" in df_raw.columns:
    df_raw = df_raw[df_raw["STATE"].isin(COASTAL_STATES)].copy()

print(f"Loaded {len(df_raw):,} coastal census tracts.")
print(f"Columns present: {list(df_raw.columns)}")

FileNotFoundError: NRI2025.csv not found.
Download the NRI Census Tract CSV from:
  https://hazards.fema.gov/nri/data-resources
Rename it to NRI2025.csv and place it next to this notebook.

## 3. Exploratory Data Analysis

In [ ]:
print(f"Shape: {df_raw.shape}")

missing = df_raw.isnull().sum()
missing = missing[missing > 0]
print("\nMissing values:")
print(missing.to_string() if not missing.empty else "  None")

if LABEL_COL in df_raw.columns:
    print(f"\n{LABEL_COL} distribution (USD):")
    print(df_raw[LABEL_COL].describe().apply(lambda x: f"${x:,.0f}").to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    clip99 = df_raw[LABEL_COL].quantile(0.99)
    df_raw[LABEL_COL].dropna().clip(upper=clip99).hist(bins=50, ax=axes[0], color="steelblue", edgecolor="none")
    axes[0].set_title("HRCN_EALB distribution (clipped at 99th pct)")
    axes[0].set_xlabel("USD")
    np.log1p(df_raw[LABEL_COL].dropna()).hist(
        bins=50, ax=axes[1], color="steelblue", edgecolor="none"
    )
    axes[1].set_title("log1p(HRCN_EALB) — normalised label shape")
    plt.tight_layout()
    plt.show()

## 4. Feature Engineering

In [ ]:
df = df_raw.copy()

for col in HURRICANE_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Engineered features
area = df.get("AREA", pd.Series(1.0, index=df.index)).replace(0, np.nan)
df["POP_DENSITY"] = (df.get("POPULATION", 0) / area).fillna(0).clip(0)

bv = df.get("BUILDVALUE", pd.Series(1.0, index=df.index)).replace(0, np.nan)
df["EXPOSURE_RATIO"] = (df.get("HRCN_EXPB", 0) / bv).fillna(0).clip(0, 1)

FEATURE_COLS = [c for c in HURRICANE_COLS if c in df.columns] + ["POP_DENSITY", "EXPOSURE_RATIO"]

# Filter to rows with a valid positive label
df_model = df.dropna(subset=[LABEL_COL]).copy()
df_model = df_model[pd.to_numeric(df_model[LABEL_COL], errors="coerce") > 0].copy()

# Normalise label to [0, 1] via log1p scaling
raw_label = pd.to_numeric(df_model[LABEL_COL], errors="coerce")
log_label = np.log1p(raw_label)
df_model["LABEL_NORM"] = (log_label / log_label.max()).clip(0, 1)

print(f"Modelling dataset: {len(df_model):,} tracts")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"\nLABEL_NORM stats:\n{df_model['LABEL_NORM'].describe().round(4)}")

## 5. Train / Test Split

In [ ]:
X = df_model[FEATURE_COLS].fillna(0)
y = df_model["LABEL_NORM"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")

## 6. Model Training

In [ ]:
def train_or_load(name, estimator):
    path = MODEL_DIR / f"{name}.joblib"
    if path.exists():
        print(f"  Loading cached: {name}")
        return name, joblib.load(path)
    print(f"  Training: {name} ...")
    pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, path)
    return name, pipe

specs = [
    ("hurricane_gbr", GradientBoostingRegressor(
        n_estimators=250, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE
    )),
    ("hurricane_rf", RandomForestRegressor(
        n_estimators=200, max_depth=16, random_state=RANDOM_STATE, n_jobs=-1
    )),
]

print("Training / loading models ...")
with ThreadPoolExecutor(max_workers=2) as ex:
    models = dict(ex.map(lambda s: train_or_load(*s), specs))

print(f"\nModels ready: {list(models.keys())}")

## 7. Model Evaluation

In [ ]:
ensemble_preds = np.mean(
    [m.predict(X_test).clip(0, 1) for m in models.values()], axis=0
)
y_arr = y_test.values

mae = mean_absolute_error(y_arr, ensemble_preds)
r2  = r2_score(y_arr, ensemble_preds)
print(f"Ensemble  MAE={mae:.4f}  R2={r2:.4f}")

for name, pipe in models.items():
    p = pipe.predict(X_test).clip(0, 1)
    print(f"  {name:20s}  MAE={mean_absolute_error(y_arr, p):.4f}  R2={r2_score(y_arr, p):.4f}")

residuals = y_arr - ensemble_preds
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(ensemble_preds, y_arr, alpha=0.3, s=8, color="steelblue")
    axes[0].plot([0,1],[0,1],"r--", linewidth=1)
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
    axes[0].set_title("Actual vs Predicted")
    axes[1].hist(residuals, bins=50, color="steelblue", edgecolor="none")
    axes[1].axvline(0, color="red", linestyle="--")
    axes[1].set_xlabel("Residual"); axes[1].set_title("Residual Distribution")
    plt.tight_layout(); plt.show()

## 8. Risk Scoring

In [ ]:
def risk_label(score):
    if score >= 0.60: return "high"
    if score >= 0.30: return "medium"
    return "low"

df_model["risk_score"] = np.mean(
    [m.predict(df_model[FEATURE_COLS].fillna(0)).clip(0, 1) for m in models.values()], axis=0
)
df_model["risk_label"] = df_model["risk_score"].apply(risk_label)

print("Risk label distribution:")
print(df_model["risk_label"].value_counts().to_string())
print(f"\nMean risk score: {df_model['risk_score'].mean():.4f}")

## 9. Geographic Output

In [ ]:
out_cols = [c for c in ID_COLS + [LAT_COL, LON_COL] if c in df_model.columns]
out_cols += [LABEL_COL, "LABEL_NORM", "risk_score", "risk_label"]

output_path = Path("hurricane_risk_scores.csv")
df_model[out_cols].to_csv(output_path, index=False)
print(f"Saved {len(df_model):,} rows -> {output_path}")
print(df_model[out_cols].head(3).to_string(index=False))

## 10. Chart: Risk Distribution

In [ ]:
if "STATEABBRV" in df_model.columns:
    state_avg = df_model.groupby("STATEABBRV")["risk_score"].mean().sort_values(ascending=False)
    colors = ["crimson" if v >= 0.6 else ("gold" if v >= 0.3 else "green") for v in state_avg]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(state_avg.index, state_avg.values, color=colors)
    ax.set_ylabel("Average Risk Score (0-1)")
    ax.set_title("Mean Hurricane Risk Score by Coastal State")
    ax.axhline(0.30, color="gold",   linestyle="--", alpha=0.7, label="Medium threshold")
    ax.axhline(0.60, color="crimson", linestyle="--", alpha=0.7, label="High threshold")
    ax.legend(); plt.tight_layout(); plt.show()

bins = np.linspace(0, 1, 40)
fig, ax = plt.subplots(figsize=(9, 4))
for lvl, clr in [("low","green"),("medium","gold"),("high","crimson")]:
    ax.hist(df_model[df_model["risk_label"]==lvl]["risk_score"],
            bins=bins, color=clr, alpha=0.7, label=lvl.capitalize())
ax.set_xlabel("Risk Score (0-1)"); ax.set_ylabel("Tracts")
ax.set_title("Hurricane Risk Score Distribution"); ax.legend()
plt.tight_layout(); plt.show()

## 11. Summary

In [ ]:
print("=" * 45)
print("  UCInsure Hurricane Risk Model — Summary")
print("=" * 45)
print(f"  Tracts scored   : {len(df_model):,}")
print(f"  Ensemble MAE    : {mae:.4f}")
print(f"  Ensemble R2     : {r2:.4f}")
print(f"  Mean risk score : {df_model['risk_score'].mean():.4f}")
print()
print("  Risk distribution:")
for lvl in ["low", "medium", "high"]:
    n = (df_model["risk_label"] == lvl).sum()
    print(f"    {lvl:8s}: {n:,} ({n/len(df_model)*100:.1f}%)")
print()
print(f"  Scored CSV -> hurricane_risk_scores.csv")
print(f"  Models     -> {MODEL_DIR}/")